# Band Selection - HistologyHSI-GB

Step 3: Reduce 699 bands to [4, 10, 20, 50, 100] using three methods:
- **PCA** - unsupervised, top-loading band per component
- **MI** - supervised, ranked by mutual information score
- **LASSO** - supervised, L1 regularization path

**Input:** `samples.h5` (67,000 pixels x 699 bands) from Google Drive  
**Output:** `bands_PCA.json`, `bands_MI.json`, `bands_LASSO.json` saved back to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import h5py
import numpy as np
import json
import time
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler

# ---- Config ----
DRIVE_DIR       = Path('/content/drive/MyDrive/HSI')
SAMPLES_FILE    = DRIVE_DIR / 'samples.h5'
BAND_COUNTS     = [4, 10, 20, 50, 100]
LASSO_N_ALPHAS  = 50
LASSO_SUBSAMPLE = 20000
RANDOM_SEED     = 42
# ----------------

print('Config OK')
print('Samples file :', SAMPLES_FILE)
print('Output dir   :', DRIVE_DIR)

In [ ]:
print('Loading samples.h5 ...')
t0 = time.time()
with h5py.File(SAMPLES_FILE, 'r') as f:
    X  = f['X'][:]
    y  = f['y'][:]
    wl = f['wavelengths'][:]
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'X  : {X.shape}  dtype={X.dtype}')
print(f'y  : {y.shape}  T={int(y.sum())}  NT={int((y==0).sum())}')
print(f'wl : {wl.shape}  {wl[0]:.1f} - {wl[-1]:.1f} nm')
assert X.shape == (67000, 699), f'Unexpected X shape: {X.shape}'
assert not np.isnan(X).any(), 'NaN found in X'
print('All checks passed.')

## 1. PCA Band Selection

Fit PCA with 100 components (the maximum band count needed). For each N, take the first N components and pick the band with the highest absolute loading from each component, deduplicating across components. This gives N interpretable wavelength positions.

In [ ]:
print('=' * 50)
print('PCA BAND SELECTION')
print('=' * 50)

MAX_COMP = max(BAND_COUNTS)
t0 = time.time()
pca = PCA(n_components=MAX_COMP, random_state=RANDOM_SEED)
pca.fit(X)
print(f'PCA fitted in {time.time()-t0:.1f}s')

explained = pca.explained_variance_ratio_
print('Explained variance by top N components:')
for n in BAND_COUNTS:
    print(f'  {n:3d} components: {100 * explained[:n].sum():.2f}%')

loadings = pca.components_  # shape: (MAX_COMP, 699)

pca_results = {}
for n in BAND_COUNTS:
    selected = []
    used = set()
    for comp_i in range(n):
        abs_load = np.abs(loadings[comp_i])
        for band_idx in np.argsort(abs_load)[::-1]:
            if int(band_idx) not in used:
                selected.append(int(band_idx))
                used.add(int(band_idx))
                break
    pca_results[str(n)] = {
        'indices': selected,
        'wavelengths_nm': [float(wl[i]) for i in selected]
    }
    wl_str = ', '.join(f'{wl[i]:.1f}' for i in selected)
    print(f'  n={n:3d}: [{wl_str}] nm')

print('PCA done.')

## 2. Mutual Information Band Selection

Compute the MI score between each of the 699 bands and the binary class label (T=1, NT=0). Rank all bands by score descending. Select top N for each band count. This is supervised - it directly measures how much each band reduces uncertainty about the class label.

In [ ]:
print('=' * 50)
print('MUTUAL INFORMATION BAND SELECTION')
print('=' * 50)
print(f'Computing MI scores ({X.shape[1]} bands, {X.shape[0]} samples) ...')
t0 = time.time()
mi_scores = mutual_info_classif(X, y, random_state=RANDOM_SEED, n_neighbors=3, n_jobs=-1)
print(f'Done in {time.time()-t0:.1f}s')

ranked_mi = np.argsort(mi_scores)[::-1]
print(f'Top band  : {wl[ranked_mi[0]]:.1f} nm  (MI score = {mi_scores[ranked_mi[0]]:.4f})')
print(f'Worst band: {wl[ranked_mi[-1]]:.1f} nm  (MI score = {mi_scores[ranked_mi[-1]]:.4f})')

mi_results = {}
for n in BAND_COUNTS:
    selected = ranked_mi[:n].tolist()
    mi_results[str(n)] = {
        'indices': [int(i) for i in selected],
        'wavelengths_nm': [float(wl[i]) for i in selected]
    }
    wl_str = ', '.join(f'{wl[i]:.1f}' for i in selected)
    print(f'  n={n:3d}: [{wl_str}] nm')

print('MI done.')

## 3. LASSO Band Selection

Fit a LASSO regression across 50 regularization strengths (alpha path). At high alpha, most band coefficients are driven to zero. As alpha decreases, more bands become non-zero. For each target N, find the alpha closest to N non-zero bands, then select the top-N by absolute coefficient magnitude. Features are standardized first. Subsampled to 20,000 pixels for speed (statistically stable).

In [ ]:
print('=' * 50)
print('LASSO BAND SELECTION')
print('=' * 50)

print('Scaling features (StandardScaler) ...')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rng = np.random.default_rng(RANDOM_SEED)
if X_scaled.shape[0] > LASSO_SUBSAMPLE:
    idx = rng.choice(X_scaled.shape[0], size=LASSO_SUBSAMPLE, replace=False)
    X_las = X_scaled[idx]
    y_las = y[idx].astype(np.float32)
    print(f'Subsampled: {LASSO_SUBSAMPLE} / {X_scaled.shape[0]} pixels used')
else:
    X_las = X_scaled
    y_las = y.astype(np.float32)

print(f'Computing LASSO path ({LASSO_N_ALPHAS} alphas) ...')
t0 = time.time()
alphas, coefs, _ = lasso_path(X_las, y_las, n_alphas=LASSO_N_ALPHAS)
print(f'Done in {time.time()-t0:.1f}s')
# coefs shape: (699, n_alphas)

n_nonzero = (coefs != 0).sum(axis=0)
print(f'Non-zero band range across path: {int(n_nonzero.min())} to {int(n_nonzero.max())}')

lasso_results = {}
for n in BAND_COUNTS:
    best_idx = int(np.argmin(np.abs(n_nonzero - n)))
    coef_vec = coefs[:, best_idx]
    actual_nz = int(n_nonzero[best_idx])
    ranked = np.argsort(np.abs(coef_vec))[::-1][:n]
    selected = sorted([int(i) for i in ranked])
    lasso_results[str(n)] = {
        'indices': selected,
        'wavelengths_nm': [float(wl[i]) for i in selected]
    }
    wl_str = ', '.join(f'{wl[i]:.1f}' for i in selected)
    print(f'  n={n:3d} (alpha={alphas[best_idx]:.5f}, actual_nz={actual_nz}): [{wl_str}] nm')

print('LASSO done.')

## 4. Save Results to Drive

In [ ]:
def save_band_json(data, path):
    with open(path, 'w') as fh:
        json.dump(data, fh, indent=2)
    kb = Path(path).stat().st_size / 1024
    print(f'Saved: {path}  ({kb:.1f} KB)')

save_band_json(pca_results,   DRIVE_DIR / 'bands_PCA.json')
save_band_json(mi_results,    DRIVE_DIR / 'bands_MI.json')
save_band_json(lasso_results, DRIVE_DIR / 'bands_LASSO.json')
print('All JSON files saved to Drive.')

## 5. Visualisation

Plot mean spectra (Tumor vs Normal) with selected band positions marked as vertical lines for each method and band count. Saved as PNG to Drive.

In [ ]:
T_mask  = y == 1
mean_T  = X[T_mask].mean(axis=0)
mean_NT = X[~T_mask].mean(axis=0)

method_list = [
    ('PCA',   pca_results,   '#2ca02c'),
    ('MI',    mi_results,    '#ff7f0e'),
    ('LASSO', lasso_results, '#9467bd'),
]

for method_name, results, color in method_list:
    fig, axes = plt.subplots(len(BAND_COUNTS), 1,
                             figsize=(14, 3 * len(BAND_COUNTS)), sharex=True)
    fig.suptitle(f'{method_name} - Selected Bands on Mean Spectrum', fontsize=13)

    for ax, n in zip(axes, BAND_COUNTS):
        ax.plot(wl, mean_T,  color='#d62728', linewidth=1.2, label='Tumor')
        ax.plot(wl, mean_NT, color='#1f77b4', linewidth=1.2, label='Normal')
        sel_wl = results[str(n)]['wavelengths_nm']
        for w in sel_wl:
            ax.axvline(w, color=color, alpha=0.7, linewidth=1.0)
        ax.set_ylabel('Reflectance', fontsize=9)
        ax.set_title(f'n={n}: {[round(w, 1) for w in sel_wl]}', fontsize=9)
        ax.legend(fontsize=8, loc='lower right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Wavelength (nm)', fontsize=10)
    plt.tight_layout()
    out_path = str(DRIVE_DIR / f'band_selection_{method_name}.png')
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

## 6. Summary

Print all selected wavelengths side by side and confirm output files.

In [ ]:
key = 'wavelengths_nm'
print('=' * 60)
print('BAND SELECTION COMPLETE')
print('=' * 60)
for n in BAND_COUNTS:
    print(f'\nn={n} bands:')
    print(f'  PCA   : {pca_results[str(n)][key]}')
    print(f'  MI    : {mi_results[str(n)][key]}')
    print(f'  LASSO : {lasso_results[str(n)][key]}')

print('\nFiles saved to Drive:')
for fp in sorted(DRIVE_DIR.glob('bands_*.json')):
    print(f'  {fp.name}  ({fp.stat().st_size / 1024:.1f} KB)')

print('\nNext steps:')
print('  1. Download bands_PCA.json, bands_MI.json, bands_LASSO.json locally')
print('  2. Place in C:\\Users\\mokas\\OneDrive\\Desktop\\HSI\\')
print('  3. git add + commit: Step 3 complete - band selection done')
print('  4. Begin Step 4: model training')